# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


FileNotFoundError: [Errno 2] No such file or directory: 'world_population.csv'

---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Justification:** 

I'd like to go with a vertical bar chart to compare the population growth of continents from 1970 to 2022. I used 1970 as the baseline (100) to make the comparison fair. The continents are shown in grey to represent its growth.

In [ ]:
# Q1 - Which continent is growing fastest?

# Calculate total population by continent and year
continent_pop = (
    df_long.groupby(['Continent', 'Year'])['Population']
    .sum()
    .reset_index()
)

# Calculate growth index (1970 = 100)
continent_pop['Growth_Index'] = (
    continent_pop['Population'] /
    continent_pop.groupby('Continent')['Population'].transform('first')
) * 100

# Get 2022 growth index and sort
growth_2022 = (
    continent_pop[continent_pop['Year'] == 2022]
    .sort_values('Growth_Index', ascending=False)
)

# Identify fastest-growing continent
fastest = growth_2022.iloc[0]['Continent']

# Create vertical bar chart
fig = px.bar(
    growth_2022,
    x='Continent',
    y='Growth_Index',
    text='Growth_Index',
    title=f'{fastest} Has the Fastest Population Growth Since 1970',
    labels={
        'Growth_Index': 'Population Growth Index (1970 = 100)',
        'Continent': 'Continent'
    }
)

fig.update_traces(
    marker_color='#2878B5',
    texttemplate='%{text:.1f}',
    textposition='outside'
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    showlegend=False,
    xaxis_showgrid=False,
    yaxis_showgrid=False
)

fig.show()

print("Fastest-growing continent:", fastest)

Fastest-growing continent: Africa


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Justification:** 

I'm going with  a horizontal bar chart because it clearly compares the population increase of the 15 countries between 1970 and 2022. African countries are represented with an accent colour while other countries are shown in grey, to make it easy to identify the African countries in the top 15.

In [ ]:

df['Population_Increase'] = df['2022 Population'] - df['1970 Population']

# Select the 15 countries with the highest population increase
top_15 = (
    df.nlargest(15, 'Population_Increase')
    .sort_values('Population_Increase', ascending=True)
    .copy()
)

# Highlight African countries
top_15['Group'] = top_15['Continent'].apply(
    lambda x: 'Africa' if x == 'Africa' else 'Other continents'
)

# Count African countries in the top 15
africa_count = (top_15['Continent'] == 'Africa').sum()

print("Number of African countries in the top 15:", africa_count)
print("\nTop 15 countries by population increase:")
print(
    top_15[
        ['Country', 'Continent', 'Population_Increase']
    ].sort_values('Population_Increase', ascending=False).to_string(index=False)
)

# Create horizontal bar chart
fig = px.bar(
    top_15,
    x='Population_Increase',
    y='Country',
    orientation='h',
    color='Group',
    text='Population_Increase',
    color_discrete_map={
        'Africa': '#0072B2',
        'Other continents': '#BDBDBD'
    },
    title=f'Top 15 Countries Added the Most People Since 1970 — {africa_count} Are in Africa',
    labels={
        'Population_Increase': 'Population Added (1970–2022)',
        'Country': '',
        'Group': 'Region'
    }
)

# Improve chart appearance
fig.update_traces(
    texttemplate='%{x:,.0f}',
    textposition='outside',
    cliponaxis=False
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    showlegend=False
)

fig.show()

Number of African countries in the top 15: 4

Top 15 countries by population increase:
      Country     Continent  Population_Increase
        India          Asia            859671872
        China          Asia            603352887
     Pakistan          Asia            176533990
      Nigeria        Africa            162971948
    Indonesia          Asia            160272945
United States North America            137961517
       Brazil South America            118943623
   Bangladesh          Asia            103644512
     Ethiopia        Africa             95071678
     DR Congo        Africa             78858479
  Philippines          Asia             78123423
       Mexico North America             77214819
        Egypt        Africa             76208117
         Iran          Asia             60100865
      Vietnam          Asia             56258007


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Justification:** 

I'm using a scatter plot to understand the relationship between land area and population using different colours representing the continents. Scales make it easier to vizualize countries with very different sizes and populations.

In [ ]:
# Q3 - Do bigger countries have more people?

# Create scatter plot
fig = px.scatter(
    df,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    hover_data={
        'Area': ':,',
        '2022 Population': ':,',
        'Continent': True
    },
    log_x=True,
    log_y=True,
    title='Country Size Does Not Always Predict Population',
    labels={
        'Area': 'Land Area (km², log scale)',
        '2022 Population': 'Population in 2022 (log scale)',
        'Continent': 'Continent'
    },
    color_discrete_sequence=px.colors.qualitative.Safe
)

# Improve chart appearance
fig.update_traces(
    marker=dict(size=9, opacity=0.75)
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    legend_title_text='Continent'
)

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Justification:** 

I'm using a box plot because it represents the distribution and spread of population density across the six continents. It helps us to compare countries with different densities easily.

In [ ]:
# Q4 - How crowded are the continents?

# Sort continents by median population density
continent_order = (
    df.groupby('Continent')['Density']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

# Create box plot
fig = px.box(
    df,
    x='Continent',
    y='Density',
    color='Continent',
    category_orders={'Continent': continent_order},
    points='outliers',
    hover_name='Country',
    hover_data={
        'Density': ':.2f',
        'Country': True,
        'Continent': False
    },
    title='Population Density Varies Widely Within Continents',
    labels={
        'Continent': '',
        'Density': 'Population Density (people per km²)'
    },
    color_discrete_sequence=px.colors.qualitative.Safe
)

# Improve chart appearance
fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    showlegend=False
)

fig.update_traces(
    marker=dict(size=5, opacity=0.7)
)

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Justification:** 

I chose a choropleth map because it represents the percentage change in population from 2000 to 2022 of countries. The geographic colour distribution will make it easy to differentiate where the population grew and where they shrink.

In [ ]:
# Q5 - Where did populations grow and where did they shrink?

# Calculate percentage population change between 2000 and 2022
df['Population_Change_Percent'] = (
    (df['2022 Population'] - df['2000 Population'])
    / df['2000 Population']
) * 100

# Count countries with population growth and decline
growth_count = (df['Population_Change_Percent'] > 0).sum()
decline_count = (df['Population_Change_Percent'] < 0).sum()

print("Countries with population growth:", growth_count)
print("Countries with population decline:", decline_count)

# Identify countries with the largest growth and decline
print("\nFive countries with the highest population growth:")
print(
    df.nlargest(5, 'Population_Change_Percent')[
        ['Country', 'Population_Change_Percent']
    ].to_string(index=False)
)

print("\nFive countries with the largest population decline:")
print(
    df.nsmallest(5, 'Population_Change_Percent')[
        ['Country', 'Population_Change_Percent']
    ].to_string(index=False)
)

# Create choropleth map
fig = px.choropleth(
    df,
    locations='CCA3',
    locationmode='ISO-3',
    color='Population_Change_Percent',
    hover_name='Country',
    hover_data={
        'Population_Change_Percent': ':.2f',
        'CCA3': False
    },
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    title='Population Growth and Decline Across Countries (2000–2022)',
    labels={
        'Population_Change_Percent': 'Population Change (%)'
    }
)

# Improve chart appearance
fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    coloraxis_colorbar_title='Change (%)'
)

fig.show()


Countries with population growth: 200
Countries with population decline: 34

Five countries with the highest population growth:
                 Country  Population_Change_Percent
                   Qatar                 317.242239
    United Arab Emirates                 188.249439
       Equatorial Guinea                 144.520327
Turks and Caicos Islands                 143.827358
                   Niger                 125.490255

Five countries with the largest population decline:
                 Country  Population_Change_Percent
Northern Mariana Islands                 -38.321840
          American Samoa                 -23.968745
               Lithuania                 -23.601880
        Marshall Islands                 -23.338374
                 Moldova                 -23.016822


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Justification:** 
I'd like to use a treemap for this question because, treemap would display the 2022 population in a order, represents both continents and their countries together in the same figure. Size of the rectangle represents population, which makes it easy to understand the share of continents and countries to the world population.

In [ ]:
# Q6 - Where do the world's 8 billion people live?

# Calculate total world population
world_population = df['2022 Population'].sum()

print("Total world population in 2022:", f"{world_population:,}")

# Create treemap
fig = px.treemap(
    df,
    path=['Continent', 'Country'],
    values='2022 Population',
    color='Continent',
    color_discrete_sequence=px.colors.qualitative.Safe,
    hover_data={
        '2022 Population': ':,'
    },
    title='Asia Contains the Largest Share of the World Population (2022)'
)

# Improve chart appearance
fig.update_traces(
    textinfo='label+percent parent',
    textfont_size=12
)

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    margin=dict(t=70, l=10, r=10, b=10)
)

fig.show()

Total world population in 2022: 7,973,413,042


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Justification:** 
I chose stacked bar chart as it clearly differentiates the world population share of Africa, Asia, Europe and the rest of the world changed between 1970 and 2022.Every bar represents full global population, changing the contribution of each group to easily understand.

In [ ]:
# Q7 - Whose share of the world is rising?

# Group the other three continents
df_long['Continent_Group'] = df_long['Continent'].apply(
    lambda x: x if x in ['Africa', 'Asia', 'Europe']
    else 'Rest of the world'
)

# Calculate population by continent group and year
share_data = (
    df_long.groupby(['Year', 'Continent_Group'])['Population']
    .sum()
    .reset_index()
)

# Calculate total world population for each year
world_totals = (
    share_data.groupby('Year')['Population']
    .sum()
    .reset_index(name='World_Population')
)

# Merge totals and calculate percentage share
share_data = share_data.merge(world_totals, on='Year')

share_data['Population_Share'] = (
    share_data['Population'] / share_data['World_Population']
) * 100

# Define the order of groups
group_order = [
    'Africa',
    'Asia',
    'Europe',
    'Rest of the world'
]

share_data['Continent_Group'] = pd.Categorical(
    share_data['Continent_Group'],
    categories=group_order,
    ordered=True
)

share_data = share_data.sort_values('Year')

# Print population shares for 1970 and 2022
print("Population share in 1970 and 2022:")
print(
    share_data[
        share_data['Year'].isin([1970, 2022])
    ][['Year', 'Continent_Group', 'Population_Share']]
    .to_string(index=False)
)

# Create 100% stacked bar chart
fig = px.bar(
    share_data,
    x='Year',
    y='Population_Share',
    color='Continent_Group',
    barmode='stack',
    category_orders={'Continent_Group': group_order},
    color_discrete_map={
        'Africa': '#0072B2',
        'Asia': '#E69F00',
        'Europe': '#009E73',
        'Rest of the world': '#BDBDBD'
    },
    title='Africa’s Share of World Population Is Rising',
    labels={
        'Year': 'Year',
        'Population_Share': 'Share of World Population (%)',
        'Continent_Group': 'Region'
    }
)

# Improve chart appearance
fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    yaxis_range=[0, 100],
    legend_title_text='Region'
)

fig.update_traces(
    hovertemplate=(
        'Year: %{x}<br>'
        'Population share: %{y:.2f}%'
        '<extra></extra>'
    )
)

fig.show()

Population share in 1970 and 2022:
 Year   Continent_Group  Population_Share
 1970            Africa          9.892551
 1970              Asia         58.062451
 1970            Europe         17.755813
 1970 Rest of the world         14.289185
 2022            Africa         17.893604
 2022              Asia         59.214081
 2022            Europe          9.320319
 2022 Rest of the world         13.571996
